## Task 1: Data Collection and Data Processing

### 1.1 Install & Import Libraries

In [1]:
import pandas as pd
import numpy as np

### 1.2 Load Dataset

In [2]:
df = pd.read_csv('data/hanoi_air_quality.csv', parse_dates=['timestamp'])

print(f'Dataset shape: {df.shape}')
print(f'Date range: {df["timestamp"].min()} → {df["timestamp"].max()}')
print(f'\nColumns: {list(df.columns)}')
df.head()

Dataset shape: (4344, 14)
Date range: 2025-11-13 00:00:00 → 2026-05-12 23:00:00

Columns: ['timestamp', 'temperature', 'humidity', 'wind_speed', 'pressure', 'hour', 'day_of_week', 'is_weekend', 'wind_direction', 'precipitation', 'cloud_cover', 'is_warning', 'aqi', 'pm25']


,timestamp,temperature,humidity,wind_speed,pressure,hour,day_of_week,is_weekend,wind_direction,precipitation,cloud_cover,is_warning,aqi,pm25
0,2025-11-13 00:00:00,20.8,81,1.361111,1015.2,0,3,0,356,0.0,100,1,113,80.6
1,2025-11-13 01:00:00,20.1,81,2.166667,1015.3,1,3,0,8,0.0,100,1,114,58.3
2,2025-11-13 02:00:00,20.1,76,2.583333,1015.0,2,3,0,9,0.0,91,1,115,42.1
3,2025-11-13 03:00:00,20.4,72,2.500000,1015.1,3,3,0,13,0.0,98,1,115,29.5
4,2025-11-13 04:00:00,20.5,70,2.361111,1015.9,4,3,0,18,0.0,99,1,116,22.0


### 1.3 Data Collection Description

Data was collected using the **Open-Meteo API** (https://open-meteo.com), a free and open-source weather API that provides hourly historical data.

- **Source:** Open-Meteo Weather Archive API + Air Quality API
- **Location:** Hanoi, Vietnam (lat: 21.0285, lon: 105.8542)
- **Period:** 6 months of hourly data
- **Collection method:** Automated API requests via Python (`collect_data.py`)

**Features collected:**

| Feature | Description | Unit | Type |
|---|---|---|---|
| `temperature` | Air temperature at 2m | °C | Compulsory |
| `humidity` | Relative humidity | % | Compulsory |
| `wind_speed` | Wind speed at 10m | m/s | Compulsory |
| `pressure` | Atmospheric pressure | hPa | Compulsory |
| `hour` | Hour of day | 0–23 | Compulsory |
| `day_of_week` | Day of week | 0–6 | Compulsory |
| `is_weekend` | Is weekend | 0/1 | Backup |
| `wind_direction` | Wind direction | degrees | Backup |
| `precipitation` | Precipitation | mm | Backup |
| `cloud_cover` | Cloud cover | % | Backup |

**Target variable:** `is_warning` - AQI > 100 (Unhealthy for Sensitive Groups)

**Validation variable (not used in model):** `pm25`, `aqi`

**Data privacy:** No personal data was collected. All data is publicly available environmental measurements.

### 1.4 Data Processing

In [3]:
# Check missing values
print('=== Missing Values ===')
print(df.isnull().sum())
print(f'\nTotal missing: {df.isnull().sum().sum()}')

=== Missing Values ===
timestamp         0
temperature       0
humidity          0
wind_speed        0
pressure          0
hour              0
day_of_week       0
is_weekend        0
wind_direction    0
precipitation     0
cloud_cover       0
is_warning        0
aqi               0
pm25              0
dtype: int64

Total missing: 0


In [4]:
# Drop rows with missing values in compulsory columns
compulsory_cols = ['temperature', 'humidity', 'wind_speed', 'pressure', 'hour', 'day_of_week', 'aqi', 'pm25']
before = len(df)
df = df.dropna(subset=compulsory_cols).reset_index(drop=True)
print(f'Dropped {before - len(df)} rows with missing values in compulsory columns')
print(f'Remaining rows: {len(df)}')

Dropped 0 rows with missing values in compulsory columns
Remaining rows: 4344


In [5]:
# Check for duplicate rows
duplicates = df.duplicated().sum()
print(f'Duplicate rows: {duplicates}')
if duplicates > 0:
    df = df.drop_duplicates().reset_index(drop=True)
    print(f'Dropped {duplicates} duplicate rows')

Duplicate rows: 0


In [6]:
# Feature encoding
# is_weekend and is_warning are already binary (0/1)
# No additional encoding needed for numerical features

print('Feature dtypes:')
print(df.dtypes)

Feature dtypes:
timestamp         datetime64[us]
temperature              float64
humidity                   int64
wind_speed               float64
pressure                 float64
hour                       int64
day_of_week                int64
is_weekend                 int64
wind_direction             int64
precipitation            float64
cloud_cover                int64
is_warning                 int64
aqi                        int64
pm25                     float64
dtype: object


In [7]:
# Normalization (Standard Scaling)
# Save original for reference, normalize for model use
from sklearn.preprocessing import StandardScaler

feature_cols = ['temperature', 'humidity', 'wind_speed', 'pressure', 'hour',
                'day_of_week', 'is_weekend', 'wind_direction', 'precipitation', 'cloud_cover']

# Only normalize continuous numerical features
cols_to_normalize = ['temperature', 'humidity', 'wind_speed', 'pressure',
                     'wind_direction', 'precipitation', 'cloud_cover']

scaler = StandardScaler()
df_normalized = df.copy()
df_normalized[cols_to_normalize] = scaler.fit_transform(df[cols_to_normalize])

print('Normalization complete!')
print(df_normalized[cols_to_normalize].describe().round(3))

Normalization complete!
       temperature  humidity  wind_speed  pressure  wind_direction  \
count     4344.000  4344.000    4344.000  4344.000        4344.000   
mean         0.000    -0.000       0.000     0.000           0.000   
std          1.000     1.000       1.000     1.000           1.000   
min         -2.665    -3.341      -2.005    -2.683          -1.481   
25%         -0.659    -0.719      -0.774    -0.704          -0.798   
50%         -0.012     0.110      -0.061     0.101          -0.004   
75%          0.657     0.869       0.674     0.777           0.265   
max          3.520     1.490       3.936     2.515           2.536   

       precipitation  cloud_cover  
count       4344.000     4344.000  
mean           0.000       -0.000  
std            1.000        1.000  
min           -0.147       -1.515  
25%           -0.147       -1.125  
50%           -0.147        0.569  
75%           -0.147        0.923  
max           26.022        0.923  


In [8]:
# Train/Test Split (80/20)
from sklearn.model_selection import train_test_split

FEATURE_COLS = ['temperature', 'humidity', 'wind_speed', 'pressure', 'hour',
                'day_of_week', 'is_weekend', 'wind_direction', 'precipitation', 'cloud_cover']
TARGET_COL   = 'is_warning'

X = df_normalized[FEATURE_COLS]
y = df_normalized[TARGET_COL]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training set : {len(X_train)} samples')
print(f'Testing set  : {len(X_test)} samples')
print(f'\nTarget distribution (train):')
print(y_train.value_counts())
print(f'\nTarget distribution (test):')
print(y_test.value_counts())

Training set : 3475 samples
Testing set  : 869 samples

Target distribution (train):
is_warning
1    2621
0     854
Name: count, dtype: int64

Target distribution (test):
is_warning
1    655
0    214
Name: count, dtype: int64
